# Bayesian Hyperparameter Tuning

Standalone notebook. Run this **independently** of the pipeline benchmarks to tune
model hyperparameters and persist the best configuration to `runs/hyperparams.json`.

The pipeline benchmark (`pipeline_benchmark.ipynb`) and main demo
(`microstructure_demo.ipynb`) will automatically load and use the saved params on
their next run — no manual copy-paste required.

## Workflow

```
bayes_tuning.ipynb          pipeline_benchmark.ipynb / microstructure_demo.ipynb
──────────────────          ──────────────────────────────────────────────────────
§1 configure scope    →     (reads scope from TUNING_SCOPE constant)
§2 build features
§3 run Optuna (100 trials)
§4 evaluate & save    →     runs/hyperparams.json   ←   §4/§6 load at startup
§5 visualise
```

**Scopes**

| Scope string | Dataset filter | Used by |
|---|---|---|
| `dp_steel` | DP/dual-phase rows only | `pipeline_benchmark.ipynb` |
| `all_alloys` | Full dataset | `microstructure_demo.ipynb` |

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.insert(0, os.path.abspath('..'))

from src.column_sanitizer import sanitize_dataframe
from src.config import PreprocessingConfig, MissingDataConfig, ScalingConfig, EncodingConfig
from src.preprocessing import FeaturePreprocessor
from src import hyperparams as hp

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               AdaBoostRegressor, ExtraTreesRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, make_scorer


# Image & morphological feature support
import re as _re
import glob
from collections import defaultdict
from src.extraction.extractor import FeatureExtractor, ExtractionConfig
from src.extraction.backbones import BackboneRegistry
from src.preprocessing.morphological import MorphologicalExtractor, MorphologyConfig
SEED     = 42
multi_r2 = make_scorer(r2_score, multioutput='uniform_average')
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('Setup complete.')
print()
print('Existing store contents:')
print(hp.summary())

## §1 — Configuration

In [ ]:
# ── Choose scope ──────────────────────────────────────────────────────────────
# "dp_steel"   → DP / dual-phase rows only  (used by pipeline_benchmark.ipynb)
# "all_alloys" → full dataset               (used by microstructure_demo.ipynb)
TUNING_SCOPE = "dp_steel"

# ── Tuning budget ─────────────────────────────────────────────────────────────
N_TRIALS = 100    # Optuna trials per model
TOP_N    = 3      # how many top models to tune

# ── CV protocols ──────────────────────────────────────────────────────────────
CV_SEARCH = RepeatedKFold(n_splits=5, n_repeats=5,  random_state=SEED)  # fast
CV_EVAL   = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)  # final

# ── Preprocessing config (must match what the pipeline uses) ──────────────────
# These defaults are overridden if a saved best_preproc exists in
# pipeline_benchmark_results.csv; otherwise set them manually to match §3.
DROP_THRESHOLD = 0.90
IMPUTE         = 'mice'       # mean | median | mice
SCALE          = 'standard'   # standard | minmax | robust | none
ENCODE         = 'onehot'     # onehot | label


# ── Feature streams ──────────────────────────────────────────────────────────
# Set BACKBONE to a cached backbone name (e.g. "dinov2_vitb14", "resnet50"),
# or None to tune on tabular features only.
BACKBONE    = "dinov2_vitb14"   # None -> tabular-only
DATA_DIR    = os.path.abspath("../data")
MORPH_CACHE = os.path.abspath("../features/morph_features_c1.npz")
print(f'Scope:          {TUNING_SCOPE}')
print(f'Trials/model:   {N_TRIALS}')
print(f'Top-N to tune:  {TOP_N}')
print(f'Preprocessing:  drop={DROP_THRESHOLD}  impute={IMPUTE}  scale={SCALE}  encode={ENCODE}')

## §2 — Load Data & Build Feature Matrix

In [ ]:
# ── Load & sanitize ───────────────────────────────────────────────────────────
df_raw = pd.read_csv('../datasets/metadata_latest.csv', header=1)
df_raw = sanitize_dataframe(df_raw)
df_raw = df_raw[df_raw['c'].notna()].copy().reset_index(drop=True)

# ── Dataset filter ────────────────────────────────────────────────────────────
if TUNING_SCOPE == 'dp_steel':
    mask = df_raw['alloy'].str.lower().str.contains(r'dp|dual.?phase', na=False, regex=True)
    df   = df_raw[mask].copy().reset_index(drop=True)
else:
    df   = df_raw.copy()

C1_TEMP    = 'cycle1_holdingtemp_degc'
C1_TIME    = 'cycle1_holdingtime_min'
TARGET_COLS = [C1_TEMP, C1_TIME]

df = df[df[C1_TEMP].notna() & df[C1_TIME].notna()].copy().reset_index(drop=True)
df['temp_time_ratio'] = df[C1_TEMP] / (df[C1_TIME] + 1e-6)

Y = df[TARGET_COLS].values.astype(np.float64)
print(f'Scope={TUNING_SCOPE}: {len(df)} rows, Y shape={Y.shape}')

# ── Feature columns ───────────────────────────────────────────────────────────
EXCLUDE_COLS = [c for c in df.columns
                if any(t in c for t in ['holdingtemp', 'holdingtime',
                                        'crate', 'rtemp', 'qtemp'])
                and any(f'cycle{n}_' in c for n in range(1, 5))]
# Keep targets out, but keep temp_time_ratio in
EXCLUDE_COLS = [c for c in EXCLUDE_COLS if c not in ['temp_time_ratio']]
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS and c not in TARGET_COLS]

COLUMN_TYPE_OVERRIDES = {
    k: v for k, v in {
        'num_cycles':          'categorical',
        'heat_treatment_type': 'categorical',
        'id':                  'unique_string',
    }.items() if k in FEATURE_COLS
}
MICE_COLS      = [c for c in ['cr', 'mo', 's', 'p', 'ni', 'al'] if c in FEATURE_COLS]
INDICATOR_COLS = [c for c in ['ti', 'nb', 'v']                   if c in FEATURE_COLS]

# ── Build preprocessor ────────────────────────────────────────────────────────
cfg = PreprocessingConfig(
    missing_data=MissingDataConfig(
        column_drop_threshold=DROP_THRESHOLD,
        row_fill_threshold=0.50,
        numeric_fill_strategy=IMPUTE if IMPUTE != 'mice' else 'median',
        categorical_fill_strategy='mode',
        mice_max_iter=10,
    ),
    scaling=ScalingConfig(method=SCALE, enabled=(SCALE != 'none')),
    encoding=EncodingConfig(categorical=ENCODE, max_categories=50),
)
prep = FeaturePreprocessor(cfg, column_types=COLUMN_TYPE_OVERRIDES,
                           mice_columns=MICE_COLS if IMPUTE == 'mice' else [],
                           indicator_columns=INDICATOR_COLS)
X = prep.fit_transform(df[FEATURE_COLS].copy())
print(f'Feature matrix: {X.shape}')
print(f'Tabular features: {X.shape}')

# ─── §2b - Image & morphological features (optional) ───────────────────────
# Replicates the alignment logic from microstructure_demo cells 18 & 24.
# Falls back to tabular-only gracefully if caches are absent.
_F_RE = _re.compile(r'_F_\d+\.[a-z]+$', _re.IGNORECASE)

def _align_to_df(cache_path, n_rows, id_series, feat_dim):
    data      = np.load(cache_path, allow_pickle=True)
    X_cache   = data['X'].astype(np.float32)
    filenames = list(data['filenames'])
    id_to_idx = defaultdict(list)
    for i, fname in enumerate(filenames):
        rid = _re.sub(r'[-\s]+', '_', _F_RE.sub('', os.path.basename(fname)).strip().lower())
        id_to_idx[rid].append(i)
    out = np.full((n_rows, feat_dim), np.nan, dtype=np.float32)
    matched = 0
    for r, row_id in enumerate(id_series.astype(str).str.strip().str.lower().str.replace(r'[-\s]+', '_', regex=True)):
        idxs = id_to_idx.get(row_id, [])
        if idxs:
            out[r] = X_cache[idxs].mean(axis=0)
            matched += 1
    col_means = np.nanmean(out, axis=0)
    nan_rows  = np.isnan(out).any(axis=1)
    out[nan_rows] = col_means
    return out, matched

X_img   = None
X_morph = None

# Image features
if BACKBONE is not None:
    _img_cache = os.path.join(DATA_DIR, f'image_cache_{BACKBONE}.npz')
    if os.path.exists(_img_cache):
        _data_check = np.load(_img_cache, allow_pickle=True)
        _feat_dim   = _data_check['X'].shape[1]
        X_img, _n_img_matched = _align_to_df(_img_cache, len(df), df['id'], _feat_dim)
        print(f'Image features   : {X_img.shape}  backbone={BACKBONE}  ({_n_img_matched}/{len(df)} matched)')
    else:
        print(f'No image cache for backbone={BACKBONE} at {_img_cache} -- skipping image features.')

# Morphological features
if os.path.exists(MORPH_CACHE):
    _cd         = np.load(MORPH_CACHE, allow_pickle=True)
    X_morph_raw = _cd['X'].astype(np.float64)
    morph_extractor = MorphologicalExtractor(MorphologyConfig(cache_path=''))
    morph_names     = morph_extractor.get_feature_names()
    X_morph_raw     = X_morph_raw[:len(df)]
    morph_prep_cfg = PreprocessingConfig(
        missing_data=MissingDataConfig(column_drop_threshold=0.95, row_fill_threshold=1.0,
                                       numeric_fill_strategy='median'),
        scaling=ScalingConfig(method='standard', enabled=True),
        encoding=EncodingConfig(categorical='onehot', max_categories=50),
    )
    morph_prep = FeaturePreprocessor(morph_prep_cfg)
    df_morph   = pd.DataFrame(X_morph_raw, columns=morph_names[:X_morph_raw.shape[1]])
    X_morph    = morph_prep.fit_transform(df_morph)
    _n_morph_matched = int((~np.isnan(X_morph_raw).any(axis=1)).sum())
    print(f'Morph features   : {X_morph.shape}  ({_n_morph_matched}/{len(df)} matched)')
else:
    print(f'No morph cache at {MORPH_CACHE} -- skipping morphological features.')

# Concatenate all streams
_parts = []
if X_img   is not None: _parts.append(X_img)
if X_morph is not None: _parts.append(X_morph)
_parts.append(X)  # tabular always last
X_full = np.concatenate(_parts, axis=1).astype(np.float64)
_stream_desc = ' + '.join(
    ([f'{X_img.shape[1]} (image)']   if X_img   is not None else []) +
    ([f'{X_morph.shape[1]} (morph)'] if X_morph is not None else []) +
    [f'{X.shape[1]} (tabular)']
)
print(f'Combined         : {X_full.shape}  [{_stream_desc}]')


## §3 — Quick Regressor Sweep

Runs a 5×5 CV sweep to identify the top-N candidates before tuning.

In [ ]:
def multi_output(est):
    return MultiOutputRegressor(est, n_jobs=-1)

CANDIDATES = {
    'RF':         RandomForestRegressor(n_estimators=200, max_features='sqrt',
                                         min_samples_leaf=3, random_state=SEED, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_features='sqrt',
                                       min_samples_leaf=3, random_state=SEED, n_jobs=-1),
    'GBR':        multi_output(GradientBoostingRegressor(
                      n_estimators=200, learning_rate=0.05, max_depth=3,
                      subsample=0.8, min_samples_leaf=5, random_state=SEED)),
    'ABR':        multi_output(AdaBoostRegressor(
                      estimator=DecisionTreeRegressor(max_depth=3),
                      n_estimators=100, random_state=SEED)),
    'SVR_rbf':    multi_output(SVR(kernel='rbf',  C=10, gamma='scale', epsilon=0.1)),
    'SVR_linear': multi_output(SVR(kernel='linear', C=1, epsilon=0.1)),
    'KNN':        multi_output(KNeighborsRegressor(n_neighbors=5)),
}

sweep_results = []
print(f'Sweeping {len(CANDIDATES)} candidates (RepeatedKFold 5x5)...')
for name, model in CANDIDATES.items():
    t0     = time.time()
    scores = cross_val_score(model, X_full, Y, cv=CV_SEARCH, scoring=multi_r2, n_jobs=-1)
    sweep_results.append({'model': name, 'mean_r2': scores.mean(), 'std_r2': scores.std()})
    print(f'  {name:<12} R²={scores.mean():.4f}±{scores.std():.4f}  ({time.time()-t0:.1f}s)')

df_sweep = (pd.DataFrame(sweep_results)
              .sort_values('mean_r2', ascending=False)
              .reset_index(drop=True))
print()
print(df_sweep.to_string(index=False))
top_models = df_sweep.head(TOP_N)['model'].tolist()
print(f'\nTop-{TOP_N} selected for Bayesian tuning: {top_models}')

## §4 — Bayesian Optimisation

In [ ]:
# ── Search space definitions ──────────────────────────────────────────────────
def build_model(trial, name):
    if name in ('RF', 'ExtraTrees'):
        p = dict(
            n_estimators    = trial.suggest_int('n_estimators', 50, 600),
            max_depth       = trial.suggest_int('max_depth', 3, 30),
            min_samples_leaf= trial.suggest_int('min_samples_leaf', 1, 10),
            max_features    = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5]),
            bootstrap       = trial.suggest_categorical('bootstrap', [True, False]),
        )
        cls = ExtraTreesRegressor if name == 'ExtraTrees' else RandomForestRegressor
        return cls(**p, random_state=SEED, n_jobs=-1)

    elif name == 'GBR':
        p = dict(
            n_estimators    = trial.suggest_int('n_estimators', 50, 500),
            learning_rate   = trial.suggest_float('learning_rate', 1e-3, 0.5, log=True),
            max_depth       = trial.suggest_int('max_depth', 2, 8),
            subsample       = trial.suggest_float('subsample', 0.4, 1.0),
            min_samples_leaf= trial.suggest_int('min_samples_leaf', 1, 20),
            max_features    = trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        )
        return multi_output(GradientBoostingRegressor(**p, random_state=SEED))

    elif name == 'ABR':
        md = trial.suggest_int('max_depth', 1, 8)
        p  = dict(n_estimators = trial.suggest_int('n_estimators', 50, 400),
                  learning_rate= trial.suggest_float('learning_rate', 0.01, 2.0, log=True))
        return multi_output(AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=md), **p, random_state=SEED))

    elif 'SVR' in name:
        kernel = 'linear' if 'linear' in name else 'rbf'
        p = dict(C      = trial.suggest_float('C', 0.01, 1000.0, log=True),
                 epsilon= trial.suggest_float('epsilon', 1e-3, 10.0, log=True))
        if kernel == 'rbf':
            p['gamma'] = trial.suggest_categorical('gamma', ['scale', 'auto'])
        return multi_output(SVR(kernel=kernel, **p))

    elif 'KNN' in name:
        p = dict(n_neighbors= trial.suggest_int('n_neighbors', 2, min(15, len(df) // 5)),
                 weights    = trial.suggest_categorical('weights', ['uniform', 'distance']),
                 p          = trial.suggest_int('p', 1, 2))
        return multi_output(KNeighborsRegressor(**p))

    raise ValueError(f'No search space for: {name}')


class _FixedTrial:
    def __init__(self, p):                       self._p = p
    def suggest_int(self, n, *a, **kw):          return self._p[n]
    def suggest_float(self, n, *a, **kw):        return self._p[n]
    def suggest_categorical(self, n, *a, **kw):  return self._p[n]


# ── Run Optuna ────────────────────────────────────────────────────────────────
studies = {}
for model_name in top_models:
    print(f'Optimising {model_name} ({N_TRIALS} trials)...')
    t0 = time.time()

    def _obj(trial, _n=model_name):
        return float(cross_val_score(
            build_model(trial, _n), X_full, Y,
            cv=CV_SEARCH, scoring=multi_r2, n_jobs=-1).mean())

    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=SEED, n_startup_trials=20),
        pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5),
    )
    study.optimize(_obj, n_trials=N_TRIALS, show_progress_bar=False)
    studies[model_name] = study
    print(f'  best R²={study.best_value:.4f}  ({time.time()-t0:.0f}s)  params={study.best_params}')
    print()

# ── Final evaluation — RepeatedKFold(5x10) ───────────────────────────────────
print('Final eval RepeatedKFold(5x10):')
print('=' * 55)
models_out = {}
for model_name in top_models:
    study  = studies[model_name]
    model  = build_model(_FixedTrial(study.best_params), model_name)
    sc     = cross_val_score(model, X_full, Y, cv=CV_EVAL, scoring=multi_r2, n_jobs=-1)
    base   = df_sweep.loc[df_sweep['model'] == model_name, 'mean_r2'].values[0]
    delta  = sc.mean() - base
    models_out[model_name] = {
        'best_cv_r2':  study.best_value,
        'tuned_cv_r2': float(sc.mean()),
        'tuned_std':   float(sc.std()),
        'delta':       float(delta),
        'params':      study.best_params,
    }
    marker = ' \u2b50' if delta > 0 else ''
    print(f'  {model_name:<12} untuned={base:.4f}  '
          f'tuned={sc.mean():.4f}\u00b1{sc.std():.4f}  \u0394={delta:+.4f}{marker}')

best_name = max(models_out, key=lambda k: models_out[k]['tuned_cv_r2'])
print(f'\nBest tuned model: {best_name}  CV R\u00b2={models_out[best_name]["tuned_cv_r2"]:.4f}')

# ── Save to hyperparameter store ──────────────────────────────────────────────
store_path = hp.save(
    scope=TUNING_SCOPE,
    models_dict=models_out,
    n_trials=N_TRIALS,
    cv_protocol=f'RepeatedKFold(n_splits=5, n_repeats=5) search / (n_repeats=10) eval',
    feature_matrix_shape=X_full.shape,
)
print(f'\nSaved to: {store_path}')
print()
print(hp.summary())

## §5 — Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — untuned vs tuned
ax     = axes[0]
names  = list(models_out.keys())
x      = np.arange(len(names))
w      = 0.35
base_r2 = [df_sweep.loc[df_sweep['model']==n, 'mean_r2'].values[0] for n in names]
tuned_r2 = [models_out[n]['tuned_cv_r2'] for n in names]
tuned_std= [models_out[n]['tuned_std']   for n in names]

ax.bar(x - w/2, base_r2,  w, label='Untuned (sweep)', alpha=0.7, color='steelblue')
ax.bar(x + w/2, tuned_r2, w, label='Bayesian-tuned',  alpha=0.85, color='#2ecc71',
       yerr=tuned_std, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Mean CV R\u00b2')
ax.set_title(f'Tuning Impact — {TUNING_SCOPE}', fontweight='bold')
ax.legend()
for xi, (u, t, e) in enumerate(zip(base_r2, tuned_r2, tuned_std)):
    col = '#27ae60' if t >= u else '#e74c3c'
    ax.text(xi + w/2, t + e + 0.005, f'{t-u:+.3f}',
            ha='center', va='bottom', fontsize=9, color=col, fontweight='bold')

# Right — optimisation history
ax2  = axes[1]
cmap = plt.cm.tab10.colors
for ci, model_name in enumerate(top_models):
    study = studies[model_name]
    vals  = [t.value for t in study.trials if t.value is not None]
    ax2.plot(range(1, len(vals)+1), np.maximum.accumulate(vals),
             label=model_name, color=cmap[ci], linewidth=1.8)
ax2.set_xlabel('Trial')
ax2.set_ylabel('Best CV R\u00b2 so far')
ax2.set_title('Optuna Optimisation History', fontweight='bold')
ax2.legend()

plt.suptitle(f'Bayesian Hyperparameter Optimisation — {TUNING_SCOPE}',
             fontsize=13, fontweight='bold')
plt.tight_layout()

out_png = f'../runs/bayes_tuning_{TUNING_SCOPE}.png'
plt.savefig(out_png, dpi=150)
plt.show()
print(f'Saved {out_png}')

# ── FAnova parameter importance ───────────────────────────────────────────────
print('\nParameter importance (FAnova):')
print('\u2500' * 55)
for model_name in top_models:
    try:
        imp = optuna.importance.get_param_importances(studies[model_name])
        print(f'\n{model_name}:')
        for k, v in imp.items():
            print(f'  {k:<24} {v:.3f}  {"\u2588" * int(v * 30)}')
    except Exception as e:
        print(f'  {model_name}: {e}')

## §6 — Verify the Saved Params

Run this cell at any time to inspect what is currently stored — no tuning required.

In [ ]:
import json as _json

print(hp.summary())
print()

# Show the winning params in full
name, params = hp.best_model(TUNING_SCOPE)
if name:
    print(f'Best model for scope=\'{TUNING_SCOPE}\': {name}')
    print(_json.dumps(params, indent=2))
else:
    print(f'No params saved for scope=\'{TUNING_SCOPE}\' yet.')